## Installation notes:

You have several alternatives, depending on what you want to run.

## Recommended: use the frozen YOLOv5 source checkout

This is the best alternative for running existing MegaDetector v5 weights such as `MDv5A` and `MDv5B`. The repository itself documents that its fallback path can use a YOLOv5 checkout on `PYTHONPATH`.

Clone the snapshot referenced by the project:

```bash
micromamba activate megadetector

mkdir -p "$HOME/git"
git clone https://github.com/agentmorris/ultralytics-yolov5.git \
  "$HOME/git/ultralytics-yolov5"
```

Add it to the environment for the current shell:

```bash
export PYTHONPATH="$HOME/git/ultralytics-yolov5:$PYTHONPATH"
```

Verify the required imports:

```bash
python -c "
from utils.general import xyxy2xywh
from utils.augmentations import letterbox
print('YOLOv5 source imports OK')
"
```

To make this persistent for the `megadetector` environment:

```bash
mkdir -p "$CONDA_PREFIX/etc/conda/activate.d"

cat > "$CONDA_PREFIX/etc/conda/activate.d/megadetector-yolov5.sh" <<'EOF'
export PYTHONPATH="$HOME/git/ultralytics-yolov5:$PYTHONPATH"
EOF
```

Then deactivate and reactivate:

```bash
micromamba deactivate
micromamba activate megadetector
```

This uses the source checkout rather than installing the broken PyPI build, while preserving the old YOLOv5 API expected by the existing MegaDetector v5 weights.

The project specifically identifies this snapshot in `TODO.md:506-509`.

## Use the current `ultralytics` package

You already have:

```text
ultralytics 8.4.118
```

The repository’s current-Ultralytics adapter imports successfully in your environment. This is appropriate for newer YOLOv8/YOLO11-style models:

```bash
uv pip install --python "$CONDA_PREFIX/bin/python" ultralytics
```

Use:

```text
model_type="ultralytics"
```

This is **not a drop-in replacement for the frozen package when loading existing MegaDetector v5 weights**. It is a separate model path for models trained/exported for the modern Ultralytics API.

## Try the generic `yolov5` PyPI package

The code mentions this as a possible alternative:

```bash
uv pip install --python "$CONDA_PREFIX/bin/python" yolov5
```

However, MegaDetector marks this path as “works, but not supported.” It may have dependency conflicts or API differences, so I would use it only for experimentation.

## Repair/build the frozen package locally

The failure comes from the legacy package build process downloading a resource during its build. A local source checkout is effectively the cleaner version of this workaround. If you want to investigate the package build itself, try:

```bash
uv pip install --python "$CONDA_PREFIX/bin/python" \
  --no-build-isolation \
  --no-deps \
  "ultralytics-yolov5==0.1.1"
```

This may still fail because the package’s `setup.py` performs its own network request; `--no-build-isolation` does not prevent that.

## My recommendation

For your immediate MegaDetector smoke test, use the source checkout:

```bash
export PYTHONPATH="$HOME/git/ultralytics-yolov5:$PYTHONPATH"
pytest -q
```

That targets the exact compatibility path required by the existing MegaDetector v5 model. Keep the installed current `ultralytics` package as well, since MegaDetector supports it for newer Ultralytics models.

---
---